<a href="https://colab.research.google.com/github/JonathanAlzate/EstructuraDatosLaboratorio/blob/main/Matriz100000%C3%97100000.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CREACIÓN DE ARCHIVO**

In [5]:
import numpy as np
import time
import os
import math

# ── Parámetros ──────────────────────────────────────────
ROWS       = 100_000
COLS       = 100_000
CHUNK_ROWS = 2_000          # filas por bloque (~25 MB de RAM por chunk)
FILE       = "matriz_bits.bin"
# ────────────────────────────────────────────────────────

COLS_PACKED = math.ceil(COLS / 8)   # bytes por fila tras empaquetar (= 12,500)

print(f"📐 Dimensiones  : {ROWS:,} x {COLS:,}")
print(f"🗜  Bytes/fila   : {COLS_PACKED:,}  ({COLS} bits → {COLS_PACKED} bytes)")
print(f"💾 Tamaño teórico: {(ROWS * COLS_PACKED) / (1024**3):.3f} GB")
print(f"🔄 Chunks        : {math.ceil(ROWS / CHUNK_ROWS)} bloques de {CHUNK_ROWS} filas\n")

rng   = np.random.default_rng(42)
start = time.time()

with open(FILE, 'wb') as f:
    for i in range(0, ROWS, CHUNK_ROWS):
        end_row = min(i + CHUNK_ROWS, ROWS)
        n_rows  = end_row - i

        # Genera chunk de 0s y 1s (uint8) → empaqueta 8 bits en 1 byte
        chunk  = rng.integers(0, 2, size=(n_rows, COLS), dtype=np.uint8)
        packed = np.packbits(chunk, axis=1)   # shape: (n_rows, 12500)
        f.write(packed.tobytes())

        # Progreso cada 10 chunks
        if (i // CHUNK_ROWS) % 10 == 0:
            pct = (end_row / ROWS) * 100
            elapsed = time.time() - start
            print(f"  ⏳ Progreso: {pct:5.1f}%  |  Tiempo transcurrido: {elapsed:.1f}s")

elapsed = time.time() - start
size_bytes = os.path.getsize(FILE)
size_gb    = size_bytes / (1024**3)
size_mb    = size_bytes / (1024**2)

print(f"\n✅ Archivo creado : {FILE}")
print(f"⏱  Tiempo total   : {elapsed:.2f} segundos")
print(f"💾 Peso resultante: {size_gb:.4f} GB  ({size_mb:,.1f} MB)  ({size_bytes:,} bytes)")



📐 Dimensiones  : 100,000 x 100,000
🗜  Bytes/fila   : 12,500  (100000 bits → 12500 bytes)
💾 Tamaño teórico: 1.164 GB
🔄 Chunks        : 50 bloques de 2000 filas

  ⏳ Progreso:   2.0%  |  Tiempo transcurrido: 0.9s
  ⏳ Progreso:  22.0%  |  Tiempo transcurrido: 10.9s
  ⏳ Progreso:  42.0%  |  Tiempo transcurrido: 21.0s
  ⏳ Progreso:  62.0%  |  Tiempo transcurrido: 31.4s
  ⏳ Progreso:  82.0%  |  Tiempo transcurrido: 40.1s

✅ Archivo creado : matriz_bits.bin
⏱  Tiempo total   : 48.59 segundos
💾 Peso resultante: 1.1642 GB  (1,192.1 MB)  (1,250,000,000 bytes)


# **LECTURA DE MATRIZ**

In [6]:
import numpy as np
import os

# ── Parámetros (deben coincidir con la escritura) ───────
ROWS        = 100_000
COLS        = 100_000
COLS_PACKED = 12_500     # ceil(100000 / 8)
FILE        = "matriz_bits.bin"
# ────────────────────────────────────────────────────────

def leer_fila(f, fila: int) -> np.ndarray:
    """Lee una fila específica sin cargar todo el archivo."""
    offset = fila * COLS_PACKED
    f.seek(offset)
    raw = np.frombuffer(f.read(COLS_PACKED), dtype=np.uint8)
    return np.unpackbits(raw)[:COLS].astype(np.bool_)

def leer_rango_filas(f, fila_inicio: int, fila_fin: int) -> np.ndarray:
    """Lee un rango de filas [fila_inicio, fila_fin)."""
    n      = fila_fin - fila_inicio
    offset = fila_inicio * COLS_PACKED
    f.seek(offset)
    raw    = np.frombuffer(f.read(n * COLS_PACKED), dtype=np.uint8)
    matrix = np.unpackbits(raw.reshape(n, COLS_PACKED), axis=1)[:, :COLS]
    return matrix.astype(np.bool_)

# ── Ejemplos de lectura ─────────────────────────────────
with open(FILE, 'rb') as f:

    # 1. Leer una sola fila
    fila_5 = leer_fila(f, 5)
    print(f"Fila 5 (primeros 20 valores) : {fila_5[:20].astype(int)}")
    print(f"Suma fila 5 (cantidad de 1s) : {fila_5.sum()}")

    # 2. Leer rango de filas (submatriz)
    sub = leer_rango_filas(f, 0, 10)
    print(f"\nSubmatriz [0:10, 0:10]:\n{sub[:, :10].astype(int)}")

    # 3. Elemento individual
    fila_x = leer_fila(f, 1_234)
    elemento = fila_x[5_678]
    print(f"\nElemento [1234, 5678] : {int(elemento)}")

# ── Info del archivo ────────────────────────────────────
size_gb = os.path.getsize(FILE) / (1024**3)
print(f"\n💾 Peso del archivo : {size_gb:.4f} GB")


Fila 5 (primeros 20 valores) : [1 1 0 0 0 1 1 1 0 1 0 0 1 0 0 1 0 1 1 0]
Suma fila 5 (cantidad de 1s) : 50105

Submatriz [0:10, 0:10]:
[[1 0 1 0 1 1 0 1 1 1]
 [1 0 0 1 1 1 0 1 0 0]
 [1 0 1 0 1 0 0 1 0 0]
 [1 1 0 1 0 0 1 0 1 1]
 [0 0 0 1 1 1 1 1 1 0]
 [1 1 0 0 0 1 1 1 0 1]
 [0 0 1 0 1 1 0 1 0 1]
 [0 0 1 1 1 1 0 0 1 1]
 [1 0 1 1 0 1 0 1 1 0]
 [0 1 0 0 1 1 1 0 0 0]]

Elemento [1234, 5678] : 1

💾 Peso del archivo : 1.1642 GB


# **DESCARGAR ARCHIVO**

In [7]:
from google.colab import files

files.download("matriz_bits.bin")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>